In [1]:
import pandas as pd
import numpy as np
import os
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA_FILE = '../Data/Processed/ML_Ready_Features.csv'
OUTPUT_FILE = '../Data/Processed/Dashboard_Predictions.csv'


In [2]:
print('Loading ML Features...')
df = pd.read_csv(DATA_FILE)

# Explicitly drop identifiers and target leakage features
leak_cols = ['restaurant_id', 'validation_status', 'internal_score', 'stability_score', 'final_priority_score', 'web_revenue', 'booking_conversion_rate', 'revenue_per_view', 'revenue_per_guest', 'velocity_ratio', 'no_show_rate', 'target_encoded', 'relative_cross_sectional_index', 'reviews_yoy_30d']
X = df.drop(columns=[c for c in leak_cols if c in df.columns] + ['target_label'])
y = df['target_label'].map({'Declining': 0, 'Stable': 1, 'Rising': 2})

cat_cols = ['region_encoded', 'cuisine_encoded']
# Ensure categoricals are integers
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype(int)

print(f'Initial features shape: {X.shape}')


Loading ML Features...
Initial features shape: (3727, 37)


In [3]:
print('--- Phase 1: Feature Pruning ---')
# 1. Spearman Rank Correlation (Multicollinearity)
print('1. Checking Spearman Correlation (>0.85)...')
num_cols = [c for c in X.columns if c not in cat_cols]
corr_matrix = X[num_cols].corr(method='spearman').abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [column for column in upper.columns if any(upper[column] > 0.85)]
print(f'Dropping highly correlated features: {to_drop_corr}')
X = X.drop(columns=to_drop_corr)

# 2. Baseline Gain Importance
print('2. Checking Baseline LightGBM Gain Importance...')
lgb_base = lgb.LGBMClassifier(random_state=42, n_estimators=50, importance_type='gain', verbose=-1)
lgb_base.fit(X, y, categorical_feature=[c for c in cat_cols if c in X.columns])
gains = pd.Series(lgb_base.feature_importances_, index=X.columns)
# Drop bottom 10%
threshold = gains.quantile(0.10)
to_drop_gain = gains[gains <= threshold].index.tolist()
print(f'Dropping bottom 10% gain features: {to_drop_gain}')
X = X.drop(columns=[c for c in to_drop_gain if c not in cat_cols]) # protect cat_cols

# 3. Permutation Importance (f1_macro)
print('3. Checking Permutation Importance (scoring=f1_macro)...')
# Train a quick RF for permutation testing
rf_perm = RandomForestClassifier(random_state=42, n_estimators=50)
rf_perm.fit(X, y)
result = permutation_importance(rf_perm, X, y, scoring='f1_macro', n_repeats=5, random_state=42)
perm_imp = pd.Series(result.importances_mean, index=X.columns)
to_drop_perm = perm_imp[perm_imp <= 0].index.tolist()
print(f'Dropping zero/negative permutation importance: {to_drop_perm}')
X = X.drop(columns=[c for c in to_drop_perm if c not in cat_cols])

print(f'Final Pruned features shape: {X.shape}')


--- Phase 1: Feature Pruning ---
1. Checking Spearman Correlation (>0.85)...
Dropping highly correlated features: ['kol_likes', 'kol_comments', 'unique_kols_used', 'kol_engagement_rate', 'gmaps_reviews', 'sentiment_variance', 'sentiment_score', 'sentiment_rating_scale', 'composite_external_rating', 'reliability_factor', 'external_quality_score', 'weighted_opportunity_gap', 'expected_kpi']
2. Checking Baseline LightGBM Gain Importance...


Dropping bottom 10% gain features: ['tiktok_views', 'tiktok_likes', 'tiktok_shares', 'facebook_likes', 'wongnai_rating', 'wongnai_reviews', 'momentum_score', 'region_encoded']
3. Checking Permutation Importance (scoring=f1_macro)...


Dropping zero/negative permutation importance: ['intent_birthday']
Final Pruned features shape: (3727, 16)


In [4]:
print('--- Phase 2: Model Benchmarking ---')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_model(model_name, X_data, y_data, is_xgb=False):
    scores = []
    
    # Needs One-Hot for XGBoost
    if is_xgb:
        X_data = pd.get_dummies(X_data, columns=[c for c in cat_cols if c in X_data.columns], drop_first=True)
        # Convert bool from get_dummies to int for xgboost
        for col in X_data.select_dtypes(include=['bool']).columns:
            X_data[col] = X_data[col].astype(int)
            
    for train_idx, test_idx in skf.split(X_data, y_data):
        X_tr, X_te = X_data.iloc[train_idx].copy(), X_data.iloc[test_idx].copy()
        y_tr, y_te = y_data.iloc[train_idx], y_data.iloc[test_idx]
        
        # Winsorization strictly inside fold based on training set clips (95th P)
        for vol_feat in ['total_revenue', 'web_views', 'total_bookings']:
            if vol_feat in X_tr.columns:
                upper = X_tr[vol_feat].quantile(0.95)
                X_tr[vol_feat] = np.where(X_tr[vol_feat] > upper, upper, X_tr[vol_feat])
                X_te[vol_feat] = np.where(X_te[vol_feat] > upper, upper, X_te[vol_feat])
        
        if model_name == 'RF':
            clf = RandomForestClassifier(random_state=42)
            clf.fit(X_tr, y_tr)
        elif model_name == 'LGBM':
            clf = lgb.LGBMClassifier(random_state=42, verbose=-1)
            clf.fit(X_tr, y_tr, categorical_feature=[c for c in cat_cols if c in X_tr.columns])
        elif model_name == 'CatBoost':
            clf = CatBoostClassifier(random_state=42, verbose=0, cat_features=[c for c in cat_cols if c in X_tr.columns])
            clf.fit(X_tr, y_tr)
        elif model_name == 'XGB':
            clf = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss')
            clf.fit(X_tr, y_tr)
            
        preds = clf.predict(X_te)
        scores.append(f1_score(y_te, preds, average='macro'))
    return np.mean(scores)

print(f"Random Forest Macro F1: {eval_model('RF', X, y):.4f}")
print(f"LightGBM Macro F1: {eval_model('LGBM', X, y):.4f}")
print(f"CatBoost Macro F1: {eval_model('CatBoost', X, y):.4f}")
print(f"XGBoost Macro F1: {eval_model('XGB', X, y, is_xgb=True):.4f}")


--- Phase 2: Model Benchmarking ---


Random Forest Macro F1: 0.7187


LightGBM Macro F1: 0.7401


CatBoost Macro F1: 0.7460


XGBoost Macro F1: 0.7393


In [5]:
print('--- Phase 3: Optuna Hyperparameter Tuning ---')

def optuna_lgbm(trial):
    params = {
        'learning_rate': trial.suggest_categorical('learning_rate', [0.01, 0.1]),
        'max_depth': trial.suggest_categorical('max_depth', [4, 10]),
        'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.6, 0.9]),
        'random_state': 42,
        'n_estimators': 100,
        'verbose': -1
    }
    
    scores = []
    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        
        for vol_feat in ['total_revenue', 'web_views', 'total_bookings']:
            if vol_feat in X_tr.columns:
                upper = X_tr[vol_feat].quantile(0.95)
                X_tr[vol_feat] = np.where(X_tr[vol_feat] > upper, upper, X_tr[vol_feat])
                X_te[vol_feat] = np.where(X_te[vol_feat] > upper, upper, X_te[vol_feat])
                
        clf = lgb.LGBMClassifier(**params)
        clf.fit(X_tr, y_tr, categorical_feature=[c for c in cat_cols if c in X.columns])
        scores.append(f1_score(y_te, clf.predict(X_te), average='macro'))
    return np.mean(scores)

study_lgbm = optuna.create_study(direction='maximize')
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lgbm.optimize(optuna_lgbm, n_trials=50)
print(f'Best LightGBM F1: {study_lgbm.best_value:.4f} with {study_lgbm.best_params}')

def optuna_cat(trial):
    params = {
        'learning_rate': trial.suggest_categorical('learning_rate', [0.01, 0.1]),
        'depth': trial.suggest_categorical('depth', [4, 10]),
        'colsample_bylevel': trial.suggest_categorical('colsample_bylevel', [0.6, 0.9]), # CatBoost
        'random_state': 42,
        'iterations': 100,
        'verbose': 0
    }
    
    scores = []
    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        
        for vol_feat in ['total_revenue', 'web_views', 'total_bookings']:
            if vol_feat in X_tr.columns:
                upper = X_tr[vol_feat].quantile(0.95)
                X_tr[vol_feat] = np.where(X_tr[vol_feat] > upper, upper, X_tr[vol_feat])
                X_te[vol_feat] = np.where(X_te[vol_feat] > upper, upper, X_te[vol_feat])
                
        clf = CatBoostClassifier(**params, cat_features=[c for c in cat_cols if c in X.columns])
        clf.fit(X_tr, y_tr)
        scores.append(f1_score(y_te, clf.predict(X_te), average='macro'))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(optuna_cat, n_trials=50)
print(f'Best CatBoost F1: {study_cat.best_value:.4f} with {study_cat.best_params}')


[I 2026-02-24 23:06:03,395] A new study created in memory with name: no-name-c9a2be86-3143-4721-91e3-601feb8d3d24


--- Phase 3: Optuna Hyperparameter Tuning ---


Best LightGBM F1: 0.7465 with {'learning_rate': 0.1, 'max_depth': 4, 'colsample_bytree': 0.6}


Best CatBoost F1: 0.7450 with {'learning_rate': 0.1, 'depth': 10, 'colsample_bylevel': 0.6}


In [6]:
print('--- Phase 4: Ensembling & Dashboard Output ---')
# Retrain best models on FULL pruned dataset
# Need to apply Winsorization to full dataset for production!
X_final = X.copy()
for vol_feat in ['total_revenue', 'web_views', 'total_bookings']:
    if vol_feat in X_final.columns:
        upper = X_final[vol_feat].quantile(0.95)
        X_final[vol_feat] = np.where(X_final[vol_feat] > upper, upper, X_final[vol_feat])

best_lgbm = lgb.LGBMClassifier(**study_lgbm.best_params, random_state=42, n_estimators=100, verbose=-1)
best_lgbm.fit(X_final, y, categorical_feature=[c for c in cat_cols if c in X_final.columns])

best_cat = CatBoostClassifier(**study_cat.best_params, random_state=42, iterations=100, verbose=0, cat_features=[c for c in cat_cols if c in X_final.columns])
best_cat.fit(X_final, y)

# Soft Voting Probabilities
lgbm_probs = best_lgbm.predict_proba(X_final)
cat_probs = best_cat.predict_proba(X_final)
ensemble_probs = (lgbm_probs + cat_probs) / 2.0

# Generate final predictions (highest probability class)
ensemble_preds = np.argmax(ensemble_probs, axis=1)
mapping = {0: 'Declining', 1: 'Stable', 2: 'Rising'}
final_labels = [mapping[p] for p in ensemble_preds]

# Extract the max probability score as the confidence
confidence_scores = np.max(ensemble_probs, axis=1)

dashboard_df = df[['restaurant_id']].copy()
dashboard_df['predicted_segment'] = final_labels
dashboard_df['probability_score'] = confidence_scores

dashboard_df.to_csv(OUTPUT_FILE, index=False)
print(f'\nâœ… Successfully Ensembled and Exported {len(dashboard_df)} records to {OUTPUT_FILE}!')
display(dashboard_df.sort_values(by='probability_score', ascending=False).head(10))


--- Phase 4: Ensembling & Dashboard Output ---



âœ… Successfully Ensembled and Exported 3727 records to Output_Data/Dashboard_Predictions.csv!


,restaurant_id,predicted_segment,probability_score
1097,3937.0,Rising,0.996883
1496,4413.0,Rising,0.996521
1669,4598.0,Rising,0.996485
889,3649.0,Rising,0.996443
936,3717.0,Rising,0.996391
157,1629.0,Rising,0.995960
1873,4854.0,Rising,0.995785
545,2884.0,Rising,0.995663
1067,3901.0,Rising,0.995390
1630,4556.0,Rising,0.995352
